In [119]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [120]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import keras
from keras.models import Sequential
from keras.optimizers import Adam, Nadam, SGD, Adamax, Adagrad
from keras.layers import Dense
from keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

import tensorflow as tf

from time import time, ctime
ctime(time())

'Thu Aug  7 15:55:01 2025'

In [121]:
# Importing the dataset
df_r = pd.read_csv(r'C:\Users\Admin\OneDrive\바탕 화면\284.실험기반 재료 물성 데이터\09.AI 모델\1.모델소스코드\Resistivity data set.csv') # Error 발생할 경우 파일 경로 확인 필수
df_r.head()

,Number,X,Y,Al,Ti,Cr,Fe,Co,Ni,Cu,...,Thickness,Resistivity,Ex_resistivity,ravg,delta,dHmix,ENavg,dEN,N,Compo
0,1,-41,-11,0.0,0.0,0.0,0.0,100.0,0.0,0.0,...,97.443489,8.166252,1.926252,0.125,0.0,0.0,1.88,0.0,1,Co
1,2,-41,-9,0.0,0.0,0.0,0.0,100.0,0.0,0.0,...,97.357119,8.159013,1.919013,0.125,0.0,0.0,1.88,0.0,1,Co
2,3,-41,-7,0.0,0.0,0.0,0.0,100.0,0.0,0.0,...,97.270749,8.107711,1.867711,0.125,0.0,0.0,1.88,0.0,1,Co
3,4,-41,-5,0.0,0.0,0.0,0.0,100.0,0.0,0.0,...,97.184379,8.100512,1.860512,0.125,0.0,0.0,1.88,0.0,1,Co
4,5,-41,-3,0.0,0.0,0.0,0.0,100.0,0.0,0.0,...,97.098008,8.093313,1.853313,0.125,0.0,0.0,1.88,0.0,1,Co


In [122]:
print('학습용 데이터의 크기: ', df_r.shape)

학습용 데이터의 크기:  (58937, 27)


In [123]:
# 학습용 데이터의 특성값을 확인한다.

print('데이터 SET의 전체 특성의 개수: ', df_r.columns.nunique())
print('__________________________________________________________________________')
print(df_r.columns.unique())

데이터 SET의 전체 특성의 개수:  27
__________________________________________________________________________
Index(['Number', 'X', 'Y', 'Al', 'Ti', 'Cr', 'Fe', 'Co', 'Ni', 'Cu', 'Zr',
       'Mo', 'W', 'Mn', 'Si', 'Mg', 'Resistance', 'Thickness', 'Resistivity',
       'Ex_resistivity', 'ravg', 'delta', 'dHmix', 'ENavg', 'dEN', 'N',
       'Compo'],
      dtype='object')


In [124]:
# 각 컬럼(특성)의 정보를 확인한다.
df_r.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58937 entries, 0 to 58936
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Number          58937 non-null  int64  
 1   X               58937 non-null  int64  
 2   Y               58937 non-null  int64  
 3   Al              58937 non-null  float64
 4   Ti              58937 non-null  float64
 5   Cr              58937 non-null  float64
 6   Fe              58937 non-null  float64
 7   Co              58937 non-null  float64
 8   Ni              58937 non-null  float64
 9   Cu              58937 non-null  float64
 10  Zr              58937 non-null  float64
 11  Mo              58937 non-null  float64
 12  W               58937 non-null  float64
 13  Mn              58937 non-null  int64  
 14  Si              58937 non-null  int64  
 15  Mg              58937 non-null  float64
 16  Resistance      58937 non-null  float64
 17  Thickness       58937 non-null 

In [125]:
df_r.describe().T

,count,mean,std,min,25%,50%,75%,max
Number,58937.0,29469.000000,17013.790745,1.000000,14735.000000,29469.000000,44203.000000,58937.000000
X,58937.0,-0.218623,21.385231,-41.000000,-17.000000,-1.000000,17.000000,41.000000
Y,58937.0,-0.000696,21.217660,-41.000000,-17.000000,0.000000,17.000000,41.000000
Al,58937.0,9.864287,16.763650,0.000000,0.000000,0.000000,15.386730,76.168290
Ti,58937.0,9.180118,15.358845,0.000000,0.000000,0.000000,15.575350,73.285380
Cr,58937.0,15.566302,23.005562,-0.803022,0.000000,0.000000,26.682000,86.389780
Fe,58937.0,9.544673,20.208080,0.000000,0.000000,0.000000,0.000000,97.245963
Co,58937.0,22.062668,37.104522,0.000000,0.000000,0.000000,30.657580,100.000000
Ni,58937.0,10.273403,19.285566,0.000000,0.000000,0.000000,14.281960,94.301490
Cu,58937.0,8.389118,17.535996,0.000000,0.000000,0.000000,2.189528,93.178460


In [126]:
df_r['Compo'].unique()

array(['Co', 'Co/Cu', 'Co/Ni', 'Zr/Cu/Al', 'Zr/Cu/Ti', 'Ni/Mo/W',
       'Co/Cr/Ti', 'Ni/Fe/Cr', 'Cu/Zr/Ti', 'Al/Cu/Zr', 'Zr/Cu/Ni/Al',
       'Al/Ti/Cr', 'Mg/Al/Zr', 'Mg/Al/Cr', 'Mg/Al/Fe', 'Mg/Al/Ti',
       'Mg/Ti/Cr', 'Ti/Cr/Fe', 'Ti/Cr/Ni', 'Cu/Zr', 'Cu/Ti', 'Ni/Zr',
       'Ni/Ti', 'Al/Ti/Fe', 'Al/Cr/Co', 'Al/Co/Mo', 'Cr/Ti/Co/Mo',
       'Cr/Ti/Co'], dtype=object)

In [127]:
df_r.isnull().any()

Number            False
X                 False
Y                 False
Al                False
Ti                False
Cr                False
Fe                False
Co                False
Ni                False
Cu                False
Zr                False
Mo                False
W                 False
Mn                False
Si                False
Mg                False
Resistance        False
Thickness         False
Resistivity       False
Ex_resistivity    False
ravg              False
delta             False
dHmix             False
ENavg             False
dEN               False
N                 False
Compo             False
dtype: bool

# 인공지능 학습 내용
- Case 1. Number, N, Compo를 제외한 22개의 피쳐값을 사용하여 Ex_resistivity 예측
- Case 2. Resistance, Thickness, Resistivity, ravg, delta, dHmix, ENavg, dEN 사용하여 Ex_resistivity 예측
Case 3. Resistivity, delta, dHmix, ENavg, dEN 사용하여 Ex_resistivity 예측
Case 4. Resistivity, delta, ENavg 사용하여 Ex_resistivity 예측
Case 5. delta 사용하여 Ex_resistivity 예측

In [128]:
# 사용하지 않는값 제외
df = df_r.drop(['Number', 'N', 'Compo', 'X', 'Y'], axis = 1)
df.head()

,Al,Ti,Cr,Fe,Co,Ni,Cu,Zr,Mo,W,...,Mg,Resistance,Thickness,Resistivity,Ex_resistivity,ravg,delta,dHmix,ENavg,dEN
0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.185,97.443489,8.166252,1.926252,0.125,0.0,0.0,1.88,0.0
1,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.185,97.357119,8.159013,1.919013,0.125,0.0,0.0,1.88,0.0
2,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.184,97.270749,8.107711,1.867711,0.125,0.0,0.0,1.88,0.0
3,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.184,97.184379,8.100512,1.860512,0.125,0.0,0.0,1.88,0.0
4,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.184,97.098008,8.093313,1.853313,0.125,0.0,0.0,1.88,0.0


In [129]:
# AI 성능 평가를 위한 함수 

def score_matrix(y_real, y_pred, X_test):
    print("MAE: ", mean_absolute_error(y_real, y_pred))
    SSE = np.sum((y_real - y_pred)**2)
    SSR = np.sum((y_pred - np.mean(y_real))**2)
    print("R2 Score :", 1-SSE/SSR)
    pred_rsq = 1 - np.sum(np.square(y_real-y_pred)) / np.var(y_real) / y_real.size
    print("Predicted R2 Score :", pred_rsq)
    p = X_test.shape[1]
    adj_r2 = 1-(SSE/SSR) * (len(y_real)-1) / (len(y_real) - p - 1)
    print("adjust R2 Score :", adj_r2)
    
data_set = []

In [130]:
from sklearn.metrics import mean_absolute_error, r2_score

def multi_score_matrix(y_true, y_pred):
    print("[MAE 및 R² Score per Component]")
    columns = ['Al', 'Ti', 'Cr', 'Fe', 'Co', 'Ni', 'Cu', 'Zr', 'Mo', 'W', 'Mn', 'Si', 'Mg']
    
    for i, col in enumerate(columns):
        mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
        r2 = r2_score(y_true[:, i], y_pred[:, i])
        print(f"{col:>3} | MAE: {mae:.4f} | R2: {r2:.4f}")


In [131]:
# 조성 예측용: 비저항성 및 실험 피처 → 조성 13종

# X: 비저항 + 실험값 (조성 제외)
X1 = np.array(df[[ 'Ex_resistivity', 'Resistance', 'Thickness',
                   'ravg', 'delta', 'dHmix', 'ENavg', 'dEN' ]])
X1 = X1.astype(float)

# y: 예측 대상은 원소 조성 13종
y1 = np.array(df[['Al', 'Ti', 'Cr', 'Fe', 'Co', 'Ni', 'Cu', 'Zr',
                  'Mo', 'W', 'Mn', 'Si', 'Mg']])
y1 = y1.astype(float)

# 학습용 데이터 분할 (같이 쓰던 구조 유지)
data_set.append(train_test_split(X1, y1, test_size=0.4, random_state=42))


# Case 2. Resistance, Thickness, ravg, delta, dHmix, ENavg, dEN 사용하여 Ex_resistivity 예측
X2 = np.array(df[["Thickness", "ravg", "delta", "dHmix", "ENavg", "dEN"]]) # 6 Features
X2 = X2.astype(float)
y2 = np.array(df["Ex_resistivity"])
y2 = y2.astype(float)
data_set.append(train_test_split(X2, y2, test_size = 0.2, random_state=42))

# Case 3. "delta", "dHmix", "ENavg", "dEN" 4가지 특성값을 사용하여 Ex_resistivity 예측
X3 = np.array(df[["delta", "dHmix", "ENavg", "dEN"]]) # 4 Features
X3 = X3.astype(float)
y3 = np.array(df["Ex_resistivity"])
y3 = y3.astype(float)
data_set.append(train_test_split(X3, y3, test_size = 0.2, random_state=38))

# Case 4. delta, ENavg 사용하여 Ex_resistivity 예측
X4 = np.array(df[[ "delta", "ENavg"]]) # 2 Features
X4 = X4.astype(float)
y4 = np.array(df["Ex_resistivity"])
y4 = y4.astype(float)
data_set.append(train_test_split(X4, y4, test_size = 0.2, random_state=38))

# Case 5. delta 사용하여 Ex_resistivity 예측
X5 = np.array(df[["delta"]]) # 1 Features
X5 = X5.astype(float)
y5 = np.array(df["Ex_resistivity"])
y5 = y5.astype(float)
data_set.append(train_test_split(X5, y5, test_size = 0.2, random_state=38))

In [153]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor

models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42),
    "MLPRegressor": MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42)
}

for name, base_model in models.items():
    print("="*60)
    print(f"🔍 모델: {name}")
    print("="*60)
    
    model = MultiOutputRegressor(base_model)
    X_train, X_test, y_train, y_test = data_set[0]  # Case 1만 사용하거나 반복문 가능
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    multi_score_matrix(y_test, y_pred)


🔍 모델: LinearRegression
[MAE 및 R² Score per Component]
 Al | MAE: 0.0710 | R2: 0.6394
 Ti | MAE: 0.0950 | R2: 0.2799
 Cr | MAE: 0.0846 | R2: 0.4682
 Fe | MAE: 0.1276 | R2: 0.3339
 Co | MAE: 0.1583 | R2: 0.7467
 Ni | MAE: 0.0945 | R2: 0.5154
 Cu | MAE: 0.0585 | R2: 0.7061
 Zr | MAE: 0.0586 | R2: 0.8796
 Mo | MAE: 0.0256 | R2: 0.6106
  W | MAE: 0.0039 | R2: 0.4226
 Mn | MAE: 0.0000 | R2: 1.0000
 Si | MAE: 0.0000 | R2: 1.0000
 Mg | MAE: 0.0023 | R2: 0.9986
🔍 모델: RandomForest
[MAE 및 R² Score per Component]
 Al | MAE: 0.0007 | R2: 0.9997
 Ti | MAE: 0.0008 | R2: 0.9989
 Cr | MAE: 0.0009 | R2: 0.9995
 Fe | MAE: 0.0004 | R2: 0.9988
 Co | MAE: 0.0004 | R2: 0.9995
 Ni | MAE: 0.0008 | R2: 0.9978
 Cu | MAE: 0.0008 | R2: 0.9948
 Zr | MAE: 0.0004 | R2: 0.9998
 Mo | MAE: 0.0001 | R2: 0.9997
  W | MAE: 0.0000 | R2: 0.9999
 Mn | MAE: 0.0000 | R2: 1.0000
 Si | MAE: 0.0000 | R2: 1.0000
 Mg | MAE: 0.0000 | R2: 1.0000
🔍 모델: XGBoost
[MAE 및 R² Score per Component]
 Al | MAE: 0.0024 | R2: 0.9990
 Ti | MAE: 0.0

# Model 2. DecisionTreeRegressor 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model = DecisionTreeRegressor()
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

# Model 3. RandomForestRegressor 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model =  RandomForestRegressor(n_estimators=25, n_jobs=-1, verbose=1)
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

# Model 4. Lasso 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model =  Lasso()
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

# Model 5. Lasso 모델을 이용한 예측

for i in range(len(data_set)):
    print("="*50)
    print("Features Case_", i+1)
    print("="*50)
    model =  Ridge()
    model.fit(data_set[i][0], data_set[i][2])
    y_pred = model.predict(data_set[i][1])
    score_matrix(data_set[i][3], y_pred, data_set[i][1])

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# 사용할 회귀 모델 5개
models = [
    ("LinearRegression", LinearRegression()),
    ("DecisionTree", DecisionTreeRegressor()),
    ("RandomForest", RandomForestRegressor()),
    ("Lasso", Lasso()),
    ("Ridge", Ridge())
]


from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# Case 1 데이터셋
X_train, X_test, y_train, y_test = data_set[0]

# 시각화 함수 정의
def plot_pred_vs_actual(y_true, y_pred, model_name):
    plt.figure(figsize=(6, 5))
    plt.scatter(y_true, y_pred, alpha=0.6)
    plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], 'r--')
    plt.title(f"[{model_name}] 예측값 vs 실제값")
    plt.xlabel("실제값 (Ex_resistivity)")
    plt.ylabel("예측값")
    plt.grid(True)
    plt.show()

def plot_residual_histogram(y_true, y_pred, model_name):
    residuals = y_true - y_pred
    plt.figure(figsize=(6, 4))
    plt.hist(residuals, bins=30, edgecolor='black')
    plt.title(f"[{model_name}] 잔차 히스토그램")
    plt.xlabel("오차 (실제 - 예측)")
    plt.ylabel("빈도")
    plt.grid(True)
    plt.show()

def plot_residual_vs_prediction(y_true, y_pred, model_name):
    residuals = y_true - y_pred
    plt.figure(figsize=(6, 4))
    plt.scatter(y_pred, residuals, alpha=0.6)
    plt.axhline(0, color='red', linestyle='--')
    plt.title(f"[{model_name}] 예측값 vs 잔차")
    plt.xlabel("예측값")
    plt.ylabel("잔차")
    plt.grid(True)
    plt.show()


for model_name, model in models:
    print("=" * 60)
    print(f"🔍 {model_name} 모델 예측 및 시각화")
    print("=" * 60)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # 평가 지표
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"MAE: {mae:.4f}")
    print(f"R2 Score: {r2:.4f}\n")

    # 시각화
    plot_pred_vs_actual(y_test, y_pred, model_name)
    plot_residual_histogram(y_test, y_pred, model_name)
    plot_residual_vs_prediction(y_test, y_pred, model_name)


In [154]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

# 최종 선택 모델
best_model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42
    )
)

# 학습
X_train, X_test, y_train, y_test = data_set[0]
best_model.fit(X_train, y_train)

# 예측
y_pred = best_model.predict(X_test)
y_pred = y_pred / y_pred.sum(axis=1, keepdims=True)  # 정규화

# 평가
multi_score_matrix(y_test, y_pred)


[MAE 및 R² Score per Component]
 Al | MAE: 0.0012 | R2: 0.9995
 Ti | MAE: 0.0013 | R2: 0.9991
 Cr | MAE: 0.0012 | R2: 0.9994
 Fe | MAE: 0.0006 | R2: 0.9996
 Co | MAE: 0.0005 | R2: 0.9999
 Ni | MAE: 0.0015 | R2: 0.9989
 Cu | MAE: 0.0011 | R2: 0.9987
 Zr | MAE: 0.0006 | R2: 0.9999
 Mo | MAE: 0.0001 | R2: 0.9996
  W | MAE: 0.0000 | R2: 0.9997
 Mn | MAE: 0.0000 | R2: 1.0000
 Si | MAE: 0.0000 | R2: 1.0000
 Mg | MAE: 0.0001 | R2: 0.9999


In [156]:
# 비교할 샘플 인덱스 선택 (예: 첫 번째 시편)
sample_index = 0

# 원소 이름
elements = ['Al', 'Ti', 'Cr', 'Fe', 'Co', 'Ni', 'Cu',
            'Zr', 'Mo', 'W', 'Mn', 'Si', 'Mg']

print(f"\n📌 샘플 {sample_index}의 실제값 vs 예측값 비교\n")
print(f"{'원소':>3} | {'실제값':>8} | {'예측값':>8} | {'오차 (절대값)':>12}")
print("-" * 42)

for i, name in enumerate(elements):
    true_val = y_test[sample_index][i]
    pred_val = y_pred[sample_index][i]
    error = abs(true_val - pred_val)
    print(f"{name:>3} | {true_val:>8.4f} | {pred_val:>8.4f} | {error:>12.4f}")



📌 샘플 0의 실제값 vs 예측값 비교

 원소 |      실제값 |      예측값 |     오차 (절대값)
------------------------------------------
 Al |   0.0000 |   0.0038 |       0.0038
 Ti |   0.0693 |   0.0649 |       0.0044
 Cr |   0.0000 |   0.0000 |       0.0000
 Fe |   0.0000 |   0.0005 |       0.0005
 Co |   0.0000 |   0.0000 |       0.0000
 Ni |   0.0000 |   0.0007 |       0.0007
 Cu |   0.6106 |   0.6108 |       0.0003
 Zr |   0.3201 |   0.3192 |       0.0008
 Mo |   0.0000 |   0.0000 |       0.0000
  W |   0.0000 |   0.0000 |       0.0000
 Mn |   0.0000 |   0.0000 |       0.0000
 Si |   0.0000 |   0.0000 |       0.0000
 Mg |   0.0000 |   0.0000 |       0.0000


In [157]:
import numpy as np

# 👉 예측 모델: 이미 학습된 RandomForest + MultiOutputRegressor 객체
# 예: model = MultiOutputRegressor(RandomForestRegressor(...)).fit(X_train, y_train)

# 🔸 원소 이름 (출력 순서)
elements = ['Al', 'Ti', 'Cr', 'Fe', 'Co', 'Ni', 'Cu',
            'Zr', 'Mo', 'W', 'Mn', 'Si', 'Mg']

# ✅ 직접 입력할 변수 8개 (순서 반드시 고정!)
# [Ex_resistivity, Resistance, Thickness, ravg, delta, dHmix, ENavg, dEN]
new_sample = np.array([[
    124.19,    # Ex_resistivity
    0.371,    # Resistance
    859,     # Thickness
    0.144,     # ravg
    0.11,   # delta
    -40.63,     # dHmix
    1.6,     # ENavg
    4      # dEN
]])

# 🔍 예측 수행
y_pred = model.predict(new_sample)

# ⚠️ 조성 정규화 (합이 1이 되도록)
y_pred = y_pred / y_pred.sum()

# 📋 결과 출력
print("\n📌 입력한 시편의 조성 예측 결과 (합 = 1)")
print("-" * 40)
for i, name in enumerate(elements):
    print(f"{name:>3} | {y_pred[0][i]:.4f}")



📌 입력한 시편의 조성 예측 결과 (합 = 1)
----------------------------------------
 Al | 0.0050
 Ti | 0.3173
 Cr | -0.2226
 Fe | -0.0012
 Co | 0.3338
 Ni | 0.0638
 Cu | 0.6362
 Zr | 0.0747
 Mo | 0.0664
  W | -0.4975
 Mn | 0.0774
 Si | 0.0774
 Mg | 0.0695


In [158]:
# 예측
y_pred = model.predict(new_sample)

# 음수 제거 + 정규화
y_pred = np.clip(y_pred, 0, None)
y_pred = y_pred / y_pred.sum()

# 결과 출력
print("\n📌 입력한 시편의 조성 예측 결과 (합 = 1)")
print("-" * 40)
for i, name in enumerate(elements):
    print(f"{name:>3} | {y_pred[0][i]:.4f}")



📌 입력한 시편의 조성 예측 결과 (합 = 1)
----------------------------------------
 Al | 0.0029
 Ti | 0.1843
 Cr | 0.0000
 Fe | 0.0000
 Co | 0.1939
 Ni | 0.0371
 Cu | 0.3696
 Zr | 0.0434
 Mo | 0.0386
  W | 0.0000
 Mn | 0.0449
 Si | 0.0449
 Mg | 0.0404
